# 04 — Audio Inference

**SONATAM** — *Symbolic Ontology & Neural Audio Transcription Architecture for Music*

This notebook demonstrates the **full inference pipeline** for a new,
unseen audio or MIDI file:

1. (Optional) Transcribe raw audio → MIDI via MT3
2. Extract dual-branch features (jSymbolic2 + semantic)
3. Add the new piece to the graph inductively
4. Predict links: genre, artist, key, era
5. Recommend similar pieces from the graph

---

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────
from pathlib import Path

import torch

from sonata.config.settings import CFG
from sonata.notebook_utils import rp
from sonata.inference import FeatureExtractor, GraphQuerier

print("Imports OK ✓")

## 1. Load trained model + graph data

In [ ]:
# Load HeteroData and trained model
hetero_data = torch.load("data/processed/hetero_data.pt", weights_only=False)
print(f"Graph: {hetero_data.node_types}")

querier = GraphQuerier(
    model_path    = "checkpoints/best_model.pt",
    hetero_data   = hetero_data,
)
print("GraphQuerier ready ✓")

## 2. Feature extraction for a new piece

The `FeatureExtractor` handles both audio (MP3/WAV/FLAC) and MIDI input.
For audio, it first transcribes to MIDI using MT3 (or basic-pitch),
then runs the dual-branch extraction.

In [ ]:
# ── Option A: from a MIDI file ──────────────────────────────────────
# input_path = Path("data/raw/lmd_matched/A/B/C/TRABCDE12345/example.mid")

# ── Option B: from raw audio ────────────────────────────────────────
# input_path = Path("data/test_audio/my_song.mp3")

# For demo, set your path here:
input_path = Path("data/raw/example.mid")  # ← replace with real file

extractor = FeatureExtractor()
features = extractor.extract(input_path)

print(f"Extracted {len(features)} features")
for k, v in list(features.items())[:10]:
    print(f"  {k}: {v}")

## 3. Predict links for the new piece

The trained GraphSAGE model predicts which **genre**, **artist**,
**key**, and **era** nodes the new piece should be connected to.

In [ ]:
predictions = querier.predict_links(features, top_k=5)

print("\n═══ Link Predictions ═══")
for edge_type, preds in predictions.items():
    print(f"\n  {edge_type}:")
    for label, score in preds:
        print(f"    {label:30s}  score={score:.4f}")

## 4. Recommend similar pieces

In [ ]:
similar = querier.recommend_similar(features, top_k=10)

print("\n═══ Similar Pieces ═══")
for rank, (piece_id, score) in enumerate(similar, 1):
    print(f"  {rank:2d}. {piece_id}  (cosine similarity: {score:.4f})")

## 5. End-to-end: audio → predictions

The `add_and_predict` method wraps the full pipeline:
audio/MIDI → features → add to graph → predict links + recommend.

In [ ]:
# result = querier.add_and_predict(input_path)
# print(result)